<a href="https://colab.research.google.com/github/Oksana0020/Diamonds-game/blob/main/Diamonds_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Diamonds game
(Search tree)

Robinson Crusoe and Friday play diamonds game on desert island using two piles of stones.

At the beginning of the game, the player chooses to play as Robinson or Friday.
The other character is controlled by AI.

##Goal of the Game
Two piles of stones, Pile A and Pile B, are placed on the ground.

Players take turns removing stones.

The game ends when both piles become (0;0)

##Rules:

On each turn a player makes one move:

1. Remove 1 to 10 stones from one pile

2. Remove the same number from both piles from 1 to 5 stones

All moves must follow these limits.


##Ideas
I want to introduce piles evolution gradually

first: small fixed piles which will be fast to test

then: (100, 100)

then: randomized piles from 80–120, generated once at game start and shown to player.

Also each player can receive two Diamond Tokens like special tools found on the island.A player may spend one token to perform a special move (for example this move can mean - remove the same number of stones from both piles 6-10)

Lucky 13 Rule - When Lucky 13 is triggered 5 stones are added to both piles, triggers multiple times during a game, may benefit or harm a player

Life on the island is unpredictable.From time to time, storms affect the piles. A storm occurs after both players have taken 5 turns each (10 total moves, then 20,30...).Storms repeats regularly.When a storm happens 3 stones are added to one random pile

If time allows to add Twisted Mode
Normal Mode:
The player who makes the move that reaches (0, 0) wins

Twisted Mode:
The player who makes the move that reaches (0, 0) loses

Possible add some dialogues in game - each character has a personality and speaks after making moves like commenting moves

Also need to display pile bars and token counts

##Constants and character labels

In [18]:
WIN_VALUE = 1
LOSS_VALUE = -1

CHAR_ROBINSON = "Robinson"
CHAR_FRIDAY = "Friday"

##CurrentBoard with initial rules

Two piles (a,b), stored sorted (a<=b).

Base legal moves:
*   Remove 1-10 stones from one pile
*   Remove 1-5 stoens from both piles (same number)

In [19]:
class CurrentBoard:
    def __init__(self, a=5, b=10):
        a = int(a); b = int(b)
        self.a, self.b = (a, b) if a <= b else (b, a)
        self.state = self.state_of_play()

    def display_board(self, game_display=False):
        print(f"Piles: ({self.a}, {self.b})")

    def state_of_play(self):
        return "FINISHED" if (self.a == 0 and self.b == 0) else "PLAYING"

    def other_player(self, forPlayer):
        return "O" if forPlayer == "X" else "X"

    def all_possible_moves(self, forPlayer="X"):
        moves = []

        # remove from pile a (max 10)
        for k in range(1, min(10, self.a) + 1):
            moves.append(CurrentBoard(self.a - k, self.b))

        # remove from pile b (max 10)
        for k in range(1, min(10, self.b) + 1):
            moves.append(CurrentBoard(self.a, self.b - k))

        # remove same from both (max 5)
        for k in range(1, min(5, self.a) + 1):
            moves.append(CurrentBoard(self.a - k, self.b - k))

        # remove duplicates
        unique, seen = [], set()
        for m in moves:
            key = (m.a, m.b)
            if key not in seen:
                seen.add(key)
                unique.append(m)

        return unique

##Checking count of moves and sample results.

In [20]:
cb = CurrentBoard(5, 10)
cb.display_board()
kids = cb.all_possible_moves()
print("Number of possible moves:", len(kids))
print("First 12 moves:", [(k.a, k.b) for k in kids[:12]])

Piles: (5, 10)
Number of possible moves: 19
First 12 moves: [(4, 10), (3, 10), (2, 10), (1, 10), (0, 10), (5, 9), (5, 8), (5, 7), (5, 6), (5, 5), (4, 5), (3, 5)]


##Add minimax tree (SearchTreeNode)

**Each node stores:**

the current board position

the player to move

all possible child positions(legal moves)


A terminal position is a leaf of the tree

(0,0) means no legal moves remain therefore the player which has to move loses

**Minimax**

The tree is evaluated using minimax algorithm.

ply depth (move number) is used:

even depth → maximizing player

odd depth → minimizing player

Terminal nodes are assigned values based on whose turn it is:

if it is the current player's turn and no moves are possible → loss (-1)

otherwise → win (+1)

How values are computed

For non-terminal nodes:

All children are evaluated recursively

Children are sorted by their minimax values

The node selects:

the maximum value at even depth (best for current player)

the minimum value at odd depth (worst for opponent)

So:

+1 means the current player can force a win

-1 means the current player will lose

In [21]:
class SearchTreeNode:
    def __init__(self, current_board, forPlayer, ply=0):
        self.board = current_board
        self.ply_depth = ply
        self.value_is_assigned = False
        self.children = []
        # expand children if game is not finished
        if current_board.state == "PLAYING":
            for move in current_board.all_possible_moves(forPlayer):
                self.children.append(
                    SearchTreeNode(move, self.board.other_player(forPlayer), self.ply_depth + 1)
                )
        else:
            self.value_is_assigned = True
            # no draw,only win or loss
            if self.ply_depth % 2 == 0:
                self.value = -1
            else:
                self.value = 1

    def min_max_value(self):
        if self.value_is_assigned:
            return self.value
        # evaluate children first
        self.children = sorted(self.children, key=lambda x: x.min_max_value())
        self.value_is_assigned = True
        # max player at even ply, min at odd ply
        if self.ply_depth % 2 == 0:
            self.value = self.children[-1].value
        else:
            self.value = self.children[0].value
        return self.value

##Checking minimax value on a small starting position

In [22]:
cb = CurrentBoard(5, 8)
st = SearchTreeNode(cb, "X")

print("Root board:", (cb.a, cb.b))
print("Children count:", len(st.children))
print("Minimax value:", st.min_max_value())

Root board: (5, 8)
Children count: 17
Minimax value: 1


##Choose the best next move

Selecting the best child:

if a child has value LOSS_VALUE, that means the next player loses there

so this is a winning move for the current player

if no such child exists, return the first child

In [23]:
def best_child_board(node: SearchTreeNode):
    if not node.children:
        return None
    node.children = sorted(node.children, key=lambda x: x.min_max_value())
    return node.children[-1].board

##Checking best move selection

In [24]:
cb = CurrentBoard(5, 8)
st = SearchTreeNode(cb, "X")
print("Root board:", (cb.a, cb.b))
print("Children count:", len(st.children))
print("Minimax value:", st.min_max_value())
best = best_child_board(st)
print("Best next board:", (best.a, best.b) if best else None)

Root board: (5, 8)
Children count: 17
Minimax value: 1
Best next board: (4, 7)


##Validate player moves if they are legal
Before creating a playable loop I need a helper that checks whether a new board is a legal result of the current board

generate all possible next boards

check if (a,b) appears among them

In [25]:
def is_legal_move(current: CurrentBoard, nxt: CurrentBoard) -> bool:
    return any((m.a, m.b) == (nxt.a, nxt.b) for m in current.all_possible_moves())

testing one legal and one illegal move from the same starting position.

In [26]:
cb = CurrentBoard(5, 8)
legal_test = CurrentBoard(5, 3)
illegal_test = CurrentBoard(1, 1)

print("Legal move test:", is_legal_move(cb, legal_test))
print("Illegal move test:", is_legal_move(cb, illegal_test))

Legal move test: True
Illegal move test: False


first playable version of the game

player enters move and AI responds using minimax

validation should catch wrong format,negative values,too-large values,moves that break the game rules,moves that make piles go below zero,

simple dialogues, bars with current state of piles